# 🏛️ Masterclass 02: Chunking Strategies (The Retrieval Architecture Layer)
### *Google Colab Ready | AI Engineer Interview Preparation (3–5 YOE)*

---

## 📌 Executive Summary
In Retrieval-Augmented Generation (RAG), **chunking is the make-or-break layer**.
A poor chunking strategy causes two fatal retrieval failure modes:
1. **Semantic Fragmentation (Under-chunking / cutting thoughts in half)**: A critical clause is split across boundaries, destroying cosine similarity for both chunks.
2. **Context Dilution / Semantic Averaging (Over-chunking / massive chunks)**: Embedding an 8,000-token chunk blurs specific details into a vague topic summary, making needle-in-a-haystack retrieval mathematically impossible.

This master notebook centralizes the complete engineering theory, mathematical formulas, and verified interactive Python implementations for:
1. **The Mathematics of Chunk Size & Overlap (Tokens vs Characters)**
2. **Strategy 1 — Fixed-Size Sliding Window (The Naive Baseline & its Traps)**
3. **Strategy 2 — Recursive Character Text Splitting (Hierarchical Separator Tree)**
4. **Strategy 3 — Document Structure-Aware Splitting (Markdown Header Hierarchy & Context Propagation)**
5. **Strategy 4 — Semantic / Embedding-Distance Chunking (Cosine Distance Spikes)**
6. **Strategy 5 — Parent-Child (Hierarchical) Chunking (The Enterprise Gold Standard)**
7. **Senior AI Interview Q&A & Architecture Decisions**


---
## ⚙️ 1. Environment Setup & Tokenizer Integration (Colab & Local)
Run the cell below to bootstrap dependencies (`tiktoken`, `langchain-text-splitters`, `numpy`) and initialize the tokenizer.


In [ ]:
import sys
import subprocess
import os
import re
import math
from typing import List, Dict, Any, Tuple

# Optional installation for Google Colab or environments missing tiktoken:
# !pip install tiktoken langchain-text-splitters numpy

# Verify or provide pure-Python fallback tokenizer for cl100k_base
try:
    import tiktoken
    tokenizer = tiktoken.get_encoding("cl100k_base")
    def count_tokens(text: str) -> int:
        return len(tokenizer.encode(text))
    print("Using official 'tiktoken' (cl100k_base tokenizer).")
except ImportError:
    # Pure-Python BPE approximation: 1 token ≈ 4 characters (approx 0.75 words)
    print("tiktoken not installed locally. Using standard token estimation (1 token ≈ 4 chars).")
    def count_tokens(text: str) -> int:
        return max(1, math.ceil(len(text) / 4))

print("Environment setup complete. Token counting function initialized.")


---
## 🔬 2. Deep Theory: Token vs Character Mathematics

### The Token Trap:
- **`len(string)`** in Python measures **Unicode characters**.
- **Embedding Models** (`text-embedding-3-small`, `text-embedding-3-large`) and LLMs operate strictly on **tokens** (Byte-Pair Encoding).
- In standard English:
  $$	ext{1 token} pprox 4	ext{ characters} pprox 0.75	ext{ words}$$
- If you configure a character-based splitter with `chunk_size = 500`, your actual chunks are only:
  $$rac{500	ext{ characters}}{4} pprox 125	ext{ tokens!}$$
  *Result*: You end up with 4x more chunks than intended, shattering paragraphs into fragments!

### The Overlap Formula:
Why do we need overlap?
If a key legal clause or technical term spans the exact boundary between Chunk $A$ and Chunk $B$, without overlap, neither chunk contains the full semantic concept.

The industry-standard overlap ratio is **$10\% - 20\%$** of chunk size:
$$	ext{chunk\_overlap} = 	ext{chunk\_size} 	imes 0.10 	ext{ to } 0.20$$

Below is a direct comparison between character-based vs token-based sizing:


In [ ]:
sample_legal_text = (
    "In accordance with Section 14.2 of the Enterprise Master Services Agreement, "
    "either party may terminate this agreement immediately upon written notice if the "
    "other party breaches any material term and fails to cure such breach within thirty (30) "
    "calendar days of receipt of written notice. Upon termination, the client shall immediately "
    "pay all accrued fees, and the service provider shall return all confidential client data."
)

char_length = len(sample_legal_text)
token_length = count_tokens(sample_legal_text)

print(f"Text Character Count: {char_length}")
print(f"Text Token Count:     {token_length}")
print(f"Actual Ratio:         {char_length / token_length:.2f} characters per token")


---
## 🪟 3. Strategy 1: Fixed-Size Sliding Window (The Naive Baseline)

### How it works:
It slides a fixed window of $N$ tokens forward by $N - 	ext{overlap}$ tokens across the text stream.

### Why it fails in Production:
It is purely mechanical. It cuts in the middle of words or sentences and completely ignores paragraph structure.
Below is the pure algorithmic implementation to understand its mechanics:


In [ ]:
def fixed_sliding_window_split(text: str, chunk_size_tokens: int = 40, overlap_tokens: int = 10) -> List[str]:
    words = text.split()
    step = chunk_size_tokens - overlap_tokens
    chunks = []
    for i in range(0, len(words), step):
        chunk_words = words[i:i + chunk_size_tokens]
        if not chunk_words:
            break
        chunks.append(" ".join(chunk_words))
        if i + chunk_size_tokens >= len(words):
            break
    return chunks

sliding_chunks = fixed_sliding_window_split(sample_legal_text, chunk_size_tokens=30, overlap_tokens=8)
print("Generated " + str(len(sliding_chunks)) + " Sliding Window Chunks:")
for idx, ch in enumerate(sliding_chunks):
    print("[Chunk " + str(idx+1) + "] (" + str(count_tokens(ch)) + " tokens):")
    print(ch)
    print()


---
## 🌲 4. Strategy 2: Recursive Character Text Splitting (The Production Workhorse)

### Deep Theory & Mathematical Logic:
Instead of cutting blindly, the **Recursive Splitter** uses an ordered hierarchy of natural language separators:
$$	ext{Separators} = [	ext{`"\n\n"' (Paragraph)}, 	ext{`"\n"' (Line/Sentence)}, 	ext{`" "' (Word)}, 	ext{`""' (Character)}]$$

### The Recursive Algorithm:
1. Try splitting the text by the highest-priority separator (`"

"`).
2. If any resulting chunk exceeds `chunk_size_tokens`, recurse on that specific piece using the next separator (`"
"`).
3. If an individual sentence still exceeds `chunk_size_tokens`, recurse using `" "` (words).
4. If a word still exceeds the limit (e.g. an unbroken 200-character URL or hash), fall back to character splitting.
5. Merge adjacent smaller chunks back together up to `chunk_size_tokens` while enforcing `chunk_overlap`.


In [ ]:
class RecursiveTokenSplitter:
    """
    Production-grade Recursive Text Splitter respecting token limits and paragraph boundaries.
    """
    def __init__(self, chunk_size: int = 60, chunk_overlap: int = 10, separators: List[str] = None):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.separators = separators or ["\n\n", "\n", " ", ""]

    def split_text(self, text: str) -> List[str]:
        return self._split(text, self.separators)

    def _split(self, text: str, separators: List[str]) -> List[str]:
        final_chunks = []
        separator = separators[-1]
        new_separators = []
        
        for i, sep in enumerate(separators):
            if sep == "":
                separator = ""
                break
            if sep in text:
                separator = sep
                new_separators = separators[i + 1:]
                break

        splits = text.split(separator) if separator != "" else list(text)
        good_splits = []
        
        for piece in splits:
            if not piece.strip():
                continue
            if count_tokens(piece) <= self.chunk_size:
                good_splits.append(piece)
            else:
                if new_separators:
                    sub_splits = self._split(piece, new_separators)
                    good_splits.extend(sub_splits)
                else:
                    good_splits.append(piece)

        # Merge good splits respecting chunk_size and chunk_overlap
        merged = []
        current_chunk = []
        current_tokens = 0

        for piece in good_splits:
            p_tokens = count_tokens(piece)
            if current_tokens + p_tokens > self.chunk_size:
                if current_chunk:
                    merged.append(separator.join(current_chunk).strip())
                    # Overlap: keep trailing pieces that fit within overlap limit
                    overlap_chunk = []
                    overlap_tokens = 0
                    for op in reversed(current_chunk):
                        op_tok = count_tokens(op)
                        if overlap_tokens + op_tok <= self.chunk_overlap:
                            overlap_chunk.insert(0, op)
                            overlap_tokens += op_tok
                        else:
                            break
                    current_chunk = overlap_chunk
                    current_tokens = overlap_tokens
            current_chunk.append(piece)
            current_tokens += p_tokens

        if current_chunk:
            merged.append(separator.join(current_chunk).strip())

        return merged

multi_paragraph_doc = (
    "Section 1: Data Encryption and Governance.\n"
    "All customer personal data must be encrypted in transit using TLS 1.3 and at rest "
    "using AES-256 GCM algorithms. Encryption keys must rotate every 90 days via KMS.\n\n"
    "Section 2: Incident Escalation Protocol.\n"
    "In the event of an unconfirmed breach, the Security Operations Center (SOC) must trigger "
    "a Sev-1 alert within 15 minutes. The CISO and Lead Architect must be notified immediately.\n\n"
    "Section 3: Audit and Compliance Logging.\n"
    "All administrative API mutations must produce immutable structured audit logs sent to a write-once "
    "storage bucket with a 7-year retention policy."
)

recursive_splitter = RecursiveTokenSplitter(chunk_size=40, chunk_overlap=8)
recursive_chunks = recursive_splitter.split_text(multi_paragraph_doc)

print(f"Generated {len(recursive_chunks)} Recursive Chunks:\n")
for idx, ch in enumerate(recursive_chunks):
    print(f"--- Chunk {idx+1} ({count_tokens(ch)} tokens) ---")
    print(ch + "\n")


---
## 📑 5. Strategy 3: Document Structure-Aware Splitting (Markdown & Header Propagation)

### The Structural Problem:
When you split a technical manual or legal code by token size alone, subsections lose their parent header.
For example, if a chunk reads:
`"- Custom engraved jewelry\n- Perishable goods"`
The vector store has **no idea** that this belongs to `# Return Policy -> ## Non-Returnable Items`!

### The Solution: Header Hierarchy Metadata Injection
1. Split along Markdown header boundaries (`#`, `##`, `###`).
2. Attach the active breadcrumb hierarchy directly into the chunk's `metadata`.
3. Optionally prepend the breadcrumb directly into the chunk content:
   `"[Product Return Policy > Non-Returnable Items]: Custom engraved jewelry..."`


In [ ]:
class MarkdownHeaderAwareSplitter:
    """
    Extracts structural Markdown sections and injects header breadcrumbs into metadata.
    """
    @staticmethod
    def split(markdown_text: str) -> List[Dict[str, Any]]:
        lines = markdown_text.splitlines()
        chunks = []
        current_h1 = None
        current_h2 = None
        current_content = []

        def save_chunk():
            if current_content:
                text_block = "\n".join(current_content).strip()
                if text_block:
                    breadcrumb = " > ".join(filter(None, [current_h1, current_h2]))
                    chunks.append({
                        "page_content": text_block,
                        "metadata": {
                            "header_1": current_h1,
                            "header_2": current_h2,
                            "breadcrumb": breadcrumb,
                            "tokens": count_tokens(text_block)
                        }
                    })

        for line in lines:
            if line.startswith("# "):
                save_chunk()
                current_h1 = line.replace("# ", "").strip()
                current_h2 = None
                current_content = []
            elif line.startswith("## "):
                save_chunk()
                current_h2 = line.replace("## ", "").strip()
                current_content = []
            else:
                current_content.append(line)
        save_chunk()
        return chunks

sample_markdown_doc = (
    "# Employee Handbook\n\n"
    "Welcome to Acme Corporation. All employees must adhere to our global code of ethics.\n\n"
    "## Remote Work Policy\n"
    "Employees may work remotely up to 3 days per week with manager sign-off. "
    "A $500 home office stipend is provided in your first paycheck.\n\n"
    "## Equipment Return\n"
    "Upon resignation, company-issued laptops and security tokens must be mailed "
    "within 5 business days using the prepaid courier box provided."
)

md_chunks = MarkdownHeaderAwareSplitter.split(sample_markdown_doc)
print(f"Extracted {len(md_chunks)} Markdown Structural Chunks:\n")
for idx, ch in enumerate(md_chunks):
    print(f"--- Section {idx+1} ---")
    print(f"Breadcrumb: {ch['metadata']['breadcrumb']}")
    print(f"Metadata:   {ch['metadata']}")
    print(f"Content:    {ch['page_content']}\n")


---
## 🧠 6. Strategy 4: Semantic / Embedding-Distance Chunking

### Deep Theory:
Instead of arbitrary token counts, **Semantic Chunking** splits text when the **semantic topic shifts**.

```
Cosine Distance Between Sentences
    ^
    |                   SPIKE (Topic Shift!)
    |                    /\
    |                   /  \
    |    _             /    \            SPIKE (Topic Shift!)
    |___/ \___________/______\___________/\____  <--- Threshold (e.g. 85th Percentile)
    +-----------------------------------------> Sentence Index
        [---- Topic A ----]  [---- Topic B ----]
```

### The Algorithm:
1. Split document into individual sentences: $[S_1, S_2, S_3, \dots, S_n]$.
2. Compute embeddings for each sentence: $[ec{e}_1, ec{e}_2, ec{e}_3, \dots, ec{e}_n]$.
3. Compute **Cosine Distance** between adjacent sentences:
   $$	ext{Distance}_i = 1 - rac{ec{e}_i \cdot ec{e}_{i+1}}{\|ec{e}_i\| \|ec{e}_{i+1}\|}$$
4. Calculate a statistical threshold (e.g., $85	ext{th}$ percentile of distance deltas).
5. Whenever distance exceeds the threshold, insert a chunk boundary!


In [ ]:
import numpy as np

def mock_semantic_sentence_embed(sentence: str) -> np.ndarray:
    vec = np.zeros(8)
    s_lower = sentence.lower()
    if any(w in s_lower for w in ["cloud", "server", "aws", "kubernetes", "infra"]):
        vec[0:4] = [0.9, 0.8, 0.7, 0.85]
    if any(w in s_lower for w in ["diet", "protein", "carbs", "nutrition", "calories"]):
        vec[4:8] = [0.85, 0.95, 0.75, 0.9]
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else np.full(8, 0.35)

def cosine_distance(v1: np.ndarray, v2: np.ndarray) -> float:
    dot = np.dot(v1, v2)
    return 1.0 - float(dot)

sentences = [
    "AWS Lambda and Kubernetes provide scalable cloud computing infrastructure.",
    "Serverless functions scale automatically with concurrent request volume.",
    "Cloud infrastructure costs must be monitored using FinOps dashboards.",
    "A high-protein diet accelerates muscle recovery after resistance training.",
    "Consuming complex carbohydrates provides sustained glycogen replenishment during workouts.",
    "Adequate hydration is critical for metabolic efficiency and electrolyte balance."
]

distances = []
for i in range(len(sentences) - 1):
    v1 = mock_semantic_sentence_embed(sentences[i])
    v2 = mock_semantic_sentence_embed(sentences[i + 1])
    dist = cosine_distance(v1, v2)
    distances.append(dist)

threshold = float(np.percentile(distances, 80))
print("Calculated Cosine Distances: " + str([round(d, 3) for d in distances]))
print("Dynamic Breakpoint Threshold: " + str(round(threshold, 3)))
print()

semantic_chunks = []
current_group = [sentences[0]]
for i, dist in enumerate(distances):
    if dist > threshold:
        print("-> Topic Shift Detected between Sentence " + str(i+1) + " and " + str(i+2))
        semantic_chunks.append(" ".join(current_group))
        current_group = [sentences[i + 1]]
    else:
        current_group.append(sentences[i + 1])
semantic_chunks.append(" ".join(current_group))

print("Formed " + str(len(semantic_chunks)) + " Cohesive Semantic Chunks:")
for idx, sc in enumerate(semantic_chunks):
    print("[Semantic Chunk " + str(idx+1) + "]:")
    print(sc)
    print()


---
## 👑 7. Strategy 5: Parent-Child (Hierarchical) Chunking (The Enterprise Gold Standard)

### The Core Paradox of RAG:
- **Small Chunks ($\sim 150–250$ tokens)**: Excel at **Retrieval**. High cosine similarity, sharp semantic focus, minimal dilution.
- **Large Chunks ($\sim 800–1500$ tokens)**: Excel at **Generation**. Provides the LLM complete paragraphs, antecedent pronouns, and surrounding context.

### The Two-Tier Architecture:
```
           +---------------------------------------------+
           |       Parent Document (e.g., 1000 tokens)   |
           |   (Stored in PostgreSQL / Redis / S3 / KV)  |
           +---------------------------------------------+
               │                     │                │
               ▼                     ▼                ▼
        +--------------+      +--------------+  +--------------+
        | Child Chunk  |      | Child Chunk  |  | Child Chunk  |
        | (200 tokens) |      | (200 tokens) |  | (200 tokens) |
        +--------------+      +--------------+  +--------------+
               │
               ▼ Embedded & Indexed in Vector DB (pgvector)
               
[User Query] ──► [Search Vector Store]
                       │
                       ▼ Matches Child Chunk #2 (High Cosine Similarity!)
                Read `metadata['parent_id']`
                       │
                       ▼
                [Fetch Parent Document from Key-Value Store]
                       │
                       ▼
                [Inject Full 1000-Token Parent into LLM Prompt]
```


In [ ]:
import uuid

class HierarchicalParentChildIndexer:
    def __init__(self, parent_size: int = 80, child_size: int = 25, child_overlap: int = 5):
        self.parent_size = parent_size
        self.child_size = child_size
        self.child_overlap = child_overlap
        self.doc_store: Dict[str, str] = {}
        self.vector_store_payloads: List[Dict[str, Any]] = []

    def index(self, full_text: str, tenant_id: str, source_id: str):
        parent_splitter = RecursiveTokenSplitter(chunk_size=self.parent_size, chunk_overlap=0)
        parents = parent_splitter.split_text(full_text)
        child_splitter = RecursiveTokenSplitter(chunk_size=self.child_size, chunk_overlap=self.child_overlap)
        for p_text in parents:
            parent_id = str(uuid.uuid4())[:8]
            self.doc_store[parent_id] = p_text
            children = child_splitter.split_text(p_text)
            for c_idx, c_text in enumerate(children):
                self.vector_store_payloads.append({
                    "child_id": f"{parent_id}_c{c_idx}",
                    "text_to_embed": c_text,
                    "metadata": {
                        "parent_id": parent_id,
                        "tenant_id": tenant_id,
                        "source": source_id,
                        "child_token_count": count_tokens(c_text)
                    }
                })

    def retrieve_context(self, matched_child_id: str) -> Dict[str, Any]:
        child_record = next(c for c in self.vector_store_payloads if c["child_id"] == matched_child_id)
        parent_id = child_record["metadata"]["parent_id"]
        parent_full_content = self.doc_store[parent_id]
        return {
            "matched_child_snippet": child_record["text_to_embed"],
            "parent_id": parent_id,
            "full_context_for_llm": parent_full_content,
            "metadata": child_record["metadata"]
        }

indexer = HierarchicalParentChildIndexer(parent_size=60, child_size=20, child_overlap=4)
indexer.index(multi_paragraph_doc, tenant_id="tenant_fintech_01", source_id="security_policy_2026.pdf")
print("Parent Document Store Count: " + str(len(indexer.doc_store)) + " parents")
print("Vector Store Child Payloads: " + str(len(indexer.vector_store_payloads)) + " child vectors indexed")
sample_child_id = indexer.vector_store_payloads[0]["child_id"]
resolved = indexer.retrieve_context(sample_child_id)
print("=== SIMULATED QUERY RETRIEVAL ===")
print("1. Matched Child Vector: " + str(resolved["matched_child_snippet"]))
print("2. Resolved Parent ID: " + str(resolved["parent_id"]))
print("3. Full Injected Context to LLM:")
print(resolved["full_context_for_llm"])


---
## 📊 8. Architectural Comparison Matrix

| Strategy | Token Granularity | Retrieval Precision | Context Richness | Computational Cost | Primary Production Use Case |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Fixed Sliding Window** | Low | Low (cuts words/sentences) | Poor | Very Low | Legacy or quick testing scripts |
| **Recursive Character** | Medium ($300-500$ tok) | High | Good | Low | General articles, blogs, essays, books |
| **Markdown Header-Aware** | Dynamic by section | High (with breadcrumbs) | Very High | Low | Technical manuals, API docs, enterprise wikis |
| **Semantic Chunking** | Dynamic by topic | Very High | High | High (many embeddings) | Unstructured transcripts, audio dictations, speeches |
| **Parent-Child (Hierarchical)** | Child: $\sim 200$, Parent: $\sim 1000$ | **Maximum (Surgical)** | **Maximum (Full Section)** | Moderate | **Enterprise Production RAG Systems** |


---
## 🎯 9. Senior AI Interview Questions & Edge Cases

### Q1: If `text-embedding-3-small` supports up to 8,191 tokens, why shouldn't we chunk documents at 8,000 tokens?
> **Answer**: Two critical reasons:
> 1. **Context Dilution / Semantic Averaging**: A single 1,536-dimensional vector representing 8,000 tokens averages out hundreds of micro-concepts into a blurry topic summary. Fine-grained queries will have low cosine similarity scores.
> 2. **Context Window Inflation & Cost**: Returning top-5 matches of 8,000 tokens sends 40,000 tokens to the LLM generation prompt, skyrocketing latency, API costs, and triggering the **"Lost in the Middle"** phenomenon.

---

### Q2: What is the "Lost in the Middle" phenomenon and how does chunking mitigate it?
> **Answer**: Research (Liu et al., 2023) demonstrates that LLMs attend best to information placed at the very beginning and very end of their input prompt. When huge, sprawling chunks are injected, relevant answers located in the middle of long contexts are frequently missed or ignored. Smaller, surgical chunks (300–500 tokens) eliminate this issue by delivering concise, high-density facts.

---

### Q3: How do you choose between Recursive Chunking and Parent-Child Chunking in production?
> **Answer**:
> - **Use Recursive Chunking**: When system architecture must remain simple, stateless, and vector-store-only, or when documents consist of short, self-contained paragraphs (e.g., FAQs, customer tickets).
> - **Use Parent-Child Chunking**: When answering questions requires deep surrounding context or chronological understanding across a full section (e.g., legal contracts, financial filings, complex medical records) without sacrificing retrieval precision.
